In [ ]:
### Load libraries and set params
library(Matrix)
library(parallel)
library(Seurat)
library(gridExtra)
library(ggplot2)
library(future)
library(ggpubr)
library(viridis)
library(openxlsx)
library(viridis)
library(scran)
library(gghighlight)
library(dplyr)
library(org.Dm.eg.db)
library(ggExtra)
library(scDblFinder)
library(patchwork)
library(RhpcBLASctl)
blas_set_num_threads(42)
library(peakRAM)
options(device=pdf)
options(future.globals.maxSize = 214748364800)
library(future)
plan("multicore", workers = 42)
library(SeuratDisk)
library(Nebulosa)

### Set directories
mainDir <- "/data/ebaird/scRNAseq/2025_2026_int/"
repDir <- paste0(mainDir, "QC_clustering/")
figDir <- paste0(repDir, "figs/")
tabDir <- paste0(repDir, "tables/")

dir.create(repDir, recursive = TRUE, showWarnings = FALSE)
dir.create(figDir, recursive = TRUE, showWarnings = FALSE)
dir.create(tabDir, recursive = TRUE, showWarnings = FALSE)

### Set colours
mycols <- c(1, '#ffffe5','#fff7bc','#fee391','#fec44f','#fe9929','#ec7014','#cc4c02','#993404','#662506')
mycols11 <- c(1, '#fee391','#fec44f','#fe9929','#ec7014','#cc4c02','#993404','#662506', "purple", "violet", "gray")
mycols13 <- c(1, '#fee391','#fec44f','#fe9929','#ec7014','#cc4c02','#993404','#662506', "purple", "violet", "gray", "blue", "green")
mycols17 <- c(1, '#fee391','#fec44f','#fe9929','#ec7014','#cc4c02','#993404','#662506', "purple", "violet", "gray", "blue", "green", rainbow(4))
mycols20 <- c("yellow", '#fee391','#fec44f','#fe9929','#ec7014','#cc4c02','#993404','#662506', "purple", "violet", "chartreuse", "blue", "green", rainbow(4), "darkslategray3", "darksalmon", "darkorchid4", "cyan")
corner <- function(x) x[1:5,1:5]
cols <- c(colorRamps::matlab.like2(20)[1:18], "deeppink2", "deeppink3", "deeppink4")
getdensity <- function(x, y, ...) {
      dens <- MASS::kde2d(x, y, ...)
      ix <- findInterval(x, dens$x)
      iy <- findInterval(y, dens$y)
      ii <- cbind(ix, iy)
      return(dens$z[ii])
}

In [ ]:
# Load filtered seurat objects to integrate

seu1 <- readRDS("/data/ebaird/scRNAseq/SCENTINELmar26/QC_clustering/filtered_regressed.rds")
seu2 <- readRDS("/data/ebaird/scRNAseq/SCENTINELsep25/QC_clustering/filtered_regressed.rds")

In [ ]:
# integration using SCTransform
# Step 1: Select features that are repeatedly variable across datasets
features <- SelectIntegrationFeatures(
  object.list = list(seu1, seu2),
  nfeatures = 2000
)

# Step 2: Prepare the SCTransform assay for integration
seu_list <- PrepSCTIntegration(
  object.list = list(seu1, seu2),
  anchor.features = features,
  verbose = TRUE
)

# Step 3: Find integration anchors
anchors <- FindIntegrationAnchors(
  object.list = seu_list,
  normalization.method = "SCT",
  anchor.features = features,  # Use the features from SelectIntegrationFeatures
  reduction = "rpca",          # or "cca"
  dims = 1:30
)

# Step 4: Integrate the datasets
integrated <- IntegrateData(
  anchorset = anchors,
  normalization.method = "SCT",
  dims = 1:30
)

# Step 5: Continue with downstream analysis
DefaultAssay(integrated) <- "integrated"
integrated <- ScaleData(integrated, verbose = FALSE)
integrated <- RunPCA(integrated, verbose = FALSE)
ElbowPlot(integrated, ndims = 35)

In [ ]:
seu <- integrated
DefaultAssay(seu) <- "RNA"
seu <- JoinLayers(seu)

# Identify genes to remove
sex_genes <- c("msl-1", "msl-2", "msl-3", "mof", "mle", "lncRNA:roX1", "lncRNA:roX2", "lncRNA:noe")
transposons <- grep("\\{\\}", rownames(seu), value = TRUE)

# Genes to keep based on RNA assay
genes_to_keep_RNA <- setdiff(rownames(seu[["RNA"]]), sex_genes)
genes_to_keep_RNA <- setdiff(genes_to_keep_RNA, transposons)

# Create new Seurat object from filtered RNA assay
seu_nosex <- CreateSeuratObject(
  counts = GetAssayData(seu, assay = "RNA", slot = "counts")[genes_to_keep_RNA, , drop = FALSE],
  meta.data = seu@meta.data
)

# Add normalized RNA data if present
rna_data <- GetAssayData(seu, assay = "RNA", slot = "data")
if (!is.null(rna_data) && nrow(rna_data) > 0) {
  seu_nosex[["RNA"]] <- SetAssayData(
    object = seu_nosex[["RNA"]],
    slot = "data",
    new.data = rna_data[genes_to_keep_RNA, , drop = FALSE]
  )
}

# Keep SCT assay if present
if ("SCT" %in% names(seu@assays)) {
  sct_genes <- rownames(seu[["SCT"]])
  genes_to_keep_SCT <- intersect(genes_to_keep_RNA, sct_genes)

  if (length(genes_to_keep_SCT) > 0) {
    seu_nosex[["SCT"]] <- CreateAssayObject(
      counts = GetAssayData(seu, assay = "SCT", slot = "counts")[genes_to_keep_SCT, , drop = FALSE]
    )

    # Add SCT normalized data
    sct_data <- GetAssayData(seu, assay = "SCT", slot = "data")
    if (!is.null(sct_data) && nrow(sct_data) > 0) {
      seu_nosex[["SCT"]] <- SetAssayData(
        object = seu_nosex[["SCT"]],
        slot = "data",
        new.data = sct_data[genes_to_keep_SCT, , drop = FALSE]
      )
    }

    # Add SCT scale.data if present
    sct_scale <- GetAssayData(seu, assay = "SCT", slot = "scale.data")
    if (!is.null(sct_scale) && nrow(sct_scale) > 0) {
      genes_in_scale <- intersect(rownames(sct_scale), genes_to_keep_SCT)
      if (length(genes_in_scale) > 0) {
        seu_nosex[["SCT"]] <- SetAssayData(
          object = seu_nosex[["SCT"]],
          slot = "scale.data",
          new.data = sct_scale[genes_in_scale, , drop = FALSE]
        )
      }
    }

    # Copy SCT variable features
    var_features <- VariableFeatures(seu, assay = "SCT")
    VariableFeatures(seu_nosex[["SCT"]]) <- intersect(var_features, genes_to_keep_SCT)
  } else {
    warning("No genes remaining in SCT assay after filtering. Skipping SCT assay transfer.")
  }
}

# Keep integrated assay if present
if ("integrated" %in% names(seu@assays)) {
  int_genes <- rownames(seu[["integrated"]])
  genes_to_keep_integrated <- intersect(genes_to_keep_RNA, int_genes)

  if (length(genes_to_keep_integrated) > 0) {
    # integrated assay usually has data and scale.data, not counts
    integrated_data <- GetAssayData(seu, assay = "integrated", slot = "data")
    seu_nosex[["integrated"]] <- CreateAssayObject(
      data = integrated_data[genes_to_keep_integrated, , drop = FALSE]
    )

    # Add scale.data if present
    integrated_scale <- GetAssayData(seu, assay = "integrated", slot = "scale.data")
    if (!is.null(integrated_scale) && nrow(integrated_scale) > 0) {
      genes_in_scale <- intersect(rownames(integrated_scale), genes_to_keep_integrated)
      if (length(genes_in_scale) > 0) {
        seu_nosex[["integrated"]] <- SetAssayData(
          object = seu_nosex[["integrated"]],
          slot = "scale.data",
          new.data = integrated_scale[genes_in_scale, , drop = FALSE]
        )
      }
    }

    # Copy variable features
    int_var_features <- VariableFeatures(seu, assay = "integrated")
    VariableFeatures(seu_nosex[["integrated"]]) <- intersect(int_var_features, genes_to_keep_integrated)
  } else {
    warning("No genes remaining in integrated assay after filtering. Skipping integrated assay transfer.")
  }
}

# Set default assay back if you want
if ("integrated" %in% names(seu_nosex@assays)) {
  DefaultAssay(seu_nosex) <- "integrated"
} else {
  DefaultAssay(seu_nosex) <- "RNA"
}

seu <- seu_nosex

In [ ]:
seu <- RunPCA(seu, npcs = 50, verbose = FALSE)
ElbowPlot(seu, ndims = 50)

In [ ]:
DefaultAssay(seu) <- "integrated"

# Plot UMAP at varying dimensions, fixed resolution at 1.0
dims_range <- 14:38
plots_dims <- list()

for (d in dims_range) {

  seu_tmp <- seu
  
  seu_tmp <- RunUMAP(seu_tmp, dims = 1:d, verbose = FALSE)
  seu_tmp <- FindNeighbors(seu_tmp, dims = 1:d, verbose = FALSE)
  seu_tmp <- FindClusters(seu_tmp, resolution = 1.0, verbose = FALSE)

  p <- DimPlot(
        seu_tmp,
        group.by = "SCT_snn_res.1",
        reduction = "umap",
        label = TRUE
      ) +
      ggtitle(paste0("dims = ", d, ", res = 1.0"))

  plots_dims[[length(plots_dims) + 1]] <- p
}

jpeg_file <- paste0(figDir, "UMAP_dims_sweep.jpeg")

jpeg(filename = jpeg_file, width = 4000, height = 4000, res = 200)

gridExtra::grid.arrange(
  grobs = plots_dims,
  ncol = 4
)

dev.off()

In [ ]:
# Plot UMAP at fixed dimensions, varying resolutions
dims_use <- 21
resolutions <- seq(0.3, 1.6, by = 0.1)

seu <- RunUMAP(seu, dims = 1:dims_use, verbose = FALSE)
seu <- FindNeighbors(seu, dims = 1:dims_use, verbose = FALSE)

seu <- FindClusters(
  seu,
  resolution = resolutions,
  verbose = FALSE
)

plots_res <- lapply(resolutions, function(r) {

  colname <- paste0("integrated_snn_res.", r)

  DimPlot(
    seu,
    group.by = colname,
    reduction = "umap",
    label = TRUE
  ) +
  ggtitle(paste0("dims = ", dims_use, ", res = ", r))

})

jpeg_file <- paste0(figDir, "UMAP_resolution_sweep.jpeg")

jpeg(filename = jpeg_file, width = 4000, height = 3000, res = 200)

gridExtra::grid.arrange(
  grobs = plots_res,
  ncol = 4
)

dev.off()

In [ ]:
# Continue with downstream analysis
seu <- RunUMAP(seu, dims = 1:21)
seu <- FindNeighbors(seu, dims = 1:21)
seu <- FindClusters(seu, resolution = 0.8)

### Save UMAPs
jpeg(paste0(figDir, "UMAP.sct_seurat_int.jpeg"), width = 900, height = 900, res = 150)
DimPlot(seu, reduction = "umap", group.by = "seurat_clusters", label = TRUE, label.size = 5)
dev.off()
DimPlot(seu, reduction = "umap", group.by = "seurat_clusters", label = TRUE, label.size = 5)

plot_height <- length(unique(seu$genotype)) * 2

jpeg(paste0(figDir, "UMAP.sct_seurat_int_split.jpeg"), width = 1500, height = 200 * plot_height, res = 150)
DimPlot(seu, reduction = "umap", group.by = "seurat_clusters", label = TRUE, label.size = 5, split.by = "condition", ncol = 2)
dev.off()
DimPlot(seu, reduction = "umap", group.by = "seurat_clusters", label = TRUE, label.size = 5, split.by = "condition", ncol = 2)

### Integrated QC feature plots
gs <- lapply(c( "nCount_RNA", "nFeature_RNA", "Ldh", "percent.mt", "percent.ribo", "S.Score", "G2M.Score", "SPARC", "pros", "elav"), function(i) FeaturePlot(seu, features=i, order=T, ncol=3, max.cutoff="q99") & scale_colour_gradientn(colours = cols))
gs[[length(gs)+1]] <- DimPlot(seu, group.by="Phase")
gs[[length(gs)+1]] <- DimPlot(seu, group.by="scDblFinder.class")
lay <- rbind(c(1,2,3),
             c(4,5,6),
             c(7,8,9),
             c(10,11,12))            
jpeg_file <- paste0(figDir, "UMAP.QC.sct.int.jpeg")
jpeg(filename = jpeg_file, width = 1800, height = 2000, res = 150)
gridExtra::grid.arrange(grobs = gs, layout_matrix = lay)
dev.off()

In [ ]:
DefaultAssay(seu) <- 'RNA'
# seu <- JoinLayers(seu)
seu <- NormalizeData(seu)
Seu <- FindVariableFeatures(seu)
seu <- ScaleData(seu)
DefaultAssay(seu) <- 'SCT'
seu <- SCTransform(seu, verbose = TRUE, return.only.var.genes = FALSE)

In [ ]:
### Save/load unfiltered object
saveRDS(seu, paste0(repDir, "integrated.rds"))
# seu <- readRDS(paste0(repDir, "integrated.sct.rds"))

In [ ]:
### Integrated QC metrics
plot <- VlnPlot(seu, features = c("nFeature_RNA", "nCount_RNA", "percent.mt", "percent.ribo"), ncol = 4, pt.size=0, group.by = "condition")
pdf(file.path(figDir, "vln_plot_QC_integrated.sct.pdf"),width = 15, height = 10)
print(plot)
dev.off()
print(plot)

# Calculate the summary statistics for nFeature, nCount and percent.mt across clusters
ncount_stats <- seu@meta.data %>%
  group_by(seurat_clusters) %>%
  summarize(
    mean_nCount = mean(nCount_RNA),
    median_nCount = median(nCount_RNA),
    min_nCount = min(nCount_RNA),
    max_nCount = max(nCount_RNA),
    sd_nCount = sd(nCount_RNA),
    n_cells = n()
  ) %>%
  ungroup()

nfeature_stats <- seu@meta.data %>%
  group_by(seurat_clusters) %>%
  summarize(
    mean_nFeature = mean(nFeature_RNA),
    median_nFeature = median(nFeature_RNA),
    min_nFeature = min(nFeature_RNA),
    max_nFeature = max(nFeature_RNA),
    sd_nFeature = sd(nFeature_RNA),
    n_cells = n()
  ) %>%
  ungroup()

percent_mt_stats <- seu@meta.data %>%
  group_by(seurat_clusters) %>%
  summarize(
    mean_percent_mt = mean(percent.mt),
    median_percent_mt = median(percent.mt),
    min_percent_mt = min(percent.mt),
    max_percent_mt = max(percent.mt),
    sd_percent_mt = sd(percent.mt),
    n_cells = n()
  ) %>%
  ungroup()

summary_stats <- left_join(ncount_stats, nfeature_stats, by = "seurat_clusters", suffix = c("_nCount", "_nFeature"))

summary_stats <- summary_stats %>%
  left_join(percent_mt_stats, by = "seurat_clusters")

sink(paste0(tabDir, "ncount_nfeature_percent.mt_stats.sct.txt"))
summary_stats
sink()

# Facet plots for nFeature, nCount and percent.mt per cluster
p1 <- ggplot(seu@meta.data, aes(x = factor(seurat_clusters), y = nCount_RNA)) +
  geom_violin(fill = "skyblue", color = "black") +
  facet_wrap(~ seurat_clusters, scales = "free") +
  labs(title = "Distribution of nCount_RNA across Clusters", 
       x = "Cluster", y = "nCount_RNA") +
  theme_minimal()

p2 <- ggplot(seu@meta.data, aes(x = factor(seurat_clusters), y = nFeature_RNA)) +
  geom_violin(fill = "lightgreen", color = "black") +
  facet_wrap(~ seurat_clusters, scales = "free") +
  labs(title = "Distribution of nFeature_RNA across Clusters", 
       x = "Cluster", y = "nFeature_RNA") +
  theme_minimal()

p3 <- ggplot(seu@meta.data, aes(x = factor(seurat_clusters), y = percent.mt)) +
  geom_violin(fill = "lightgreen", color = "black") +
  facet_wrap(~ seurat_clusters, scales = "free") +
  labs(title = "Distribution of percent.mt across Clusters", 
       x = "Cluster", y = "percent.mt") +
  theme_minimal()

pdf(file = paste0(figDir, "vln.plots.nfeature.ncount.percentmt.per.cluster.sct.pdf"), width = 18, height = 10)
print(p1)
print(p2)
print(p3)
dev.off()

print(p1)
print(p2)
print(p3)

In [ ]:
## Find markers
DefaultAssay(seu) <- 'SCT'
Idents(seu) <- 'seurat_clusters'
all.markers <- FindAllMarkers(seu, only.pos = TRUE, min.pct = 0.2, logfc.threshold = 0.25)
write.csv(all.markers,file=paste0(tabDir,'allMarkers.sct.renorm.csv'))
all.markers %>%
        group_by(cluster) %>%
        slice_max(n = 5, order_by = avg_log2FC) -> top
write.csv(top,file=paste0(tabDir,'top5Markers.sct.renorm.csv'))

genes <- unique(top$gene)
n_clusters <- length(unique(seu$seurat_clusters))

DefaultAssay(seu) <- 'SCT'
seu <- ScaleData(seu)
jpeg(paste0(figDir,'top5markers.heatmap.sct.renorm.jpeg'), quality = 100, width = 2500, height = 1300, res = 150)
print(DoHeatmap(subset(seu, downsample = 500), features = genes, ) + NoLegend())
dev.off()


# # DotPlot
DefaultAssay(seu) <- 'SCT'
plot_width <- max(8, 0.2 * length(genes))
plot_height <- max(4, 0.3 * n_clusters)
p <- DotPlot(seu, features = genes, dot.scale = 6) + 
  RotatedAxis() +
  theme(
    axis.text.x = element_text(size = 8)
  ) +
  scale_color_gradientn(
    colours = c("grey", "forestgreen")
  ) +
  ggtitle("Top 5 Markers per cluster")
ggsave(filename = paste0(figDir, 'top5markers.dotplot.sct.renorm.jpeg'), plot = p, width = plot_width, height = plot_height, dpi = 300)

In [ ]:
seu

In [ ]:
?plot_density

In [ ]:
jpeg(paste0(figDir, "density_plot_coexpress.sct.jpeg"), width = 3000, height = 1000, res = 150)
plot_density(seu, c("pros", "elav"), joint = TRUE, combine = TRUE) & scale_colour_gradientn(colours = cols)
dev.off()

In [ ]:
FeaturePlot(seu, features=c("elav", "pros"), blend = TRUE, slot = "data", blend.threshold = 0.05, combine = FALSE, pt.size = TRUE, cols = c("gray90", "blue", "red"))

In [ ]:
gs <- lapply(c( "Imp", "Syp", "Eip93F", "Ldh", "Tep4", "Gadd45", "HmgZ", "Bre1", "cas", "ase", "pros", "elav"), function(i) FeaturePlot(seu, features=i, order=T, ncol=3, max.cutoff="q99") & scale_colour_gradientn(colours = cols))
lay <- rbind(c(1,2,3),
             c(4,5,6),
             c(7,8,9),
             c(10,11,12))            
jpeg_file <- paste0(figDir, "UMAP.marker.genes.integrated.jpeg")
jpeg(filename = jpeg_file, width = 1800, height = 2000, res = 150)
gridExtra::grid.arrange(grobs = gs, layout_matrix = lay)
dev.off()

In [ ]:
### DE between specific clusters that potentially need merging based on heatmap and dotplot visualisation

cluster_pairs <- list(
  "11_vs_19" = c("11", "19")
)

for (pair_name in names(cluster_pairs)) {
  clusters <- cluster_pairs[[pair_name]]
  
  cluster_subset <- subset(seu, idents = clusters)
  Idents(cluster_subset) <- "seurat_clusters"
  
  de_results <- FindMarkers(
    object = cluster_subset,
    ident.1 = clusters[1],
    ident.2 = clusters[2],
    min.pct = 0.25,
    logfc.threshold = 0.25
  )
  
  filtered_results <- subset(de_results, p_val < 0.05 & abs(avg_log2FC) > 0.5)
  write.csv(filtered_results, file.path(tabDir, paste0("DE_results_cluster_", pair_name, ".csv")))
  
  top_genes <- head(rownames(filtered_results[order(-abs(filtered_results$avg_log2FC)), ]), 6)
  
  plots <- list()
  for (gene in top_genes) {
    p <- VlnPlot(cluster_subset, features = gene, group.by = "seurat_clusters", pt.size = 0) +
      ggtitle(paste0("Cluster ", pair_name, " - ", gene)) + theme(plot.title = element_text(size = 10))
    plots[[gene]] <- p
  }
  
  plots <- wrap_plots(plots, ncol = 2)
  
  pdf(file.path(figDir, paste0("violin_plots_cluster_comparison_", pair_name, ".pdf")))
  print(plots)
  dev.off()
}

In [ ]:
## Manually merge clusters based on DE results and visual inspection of markers
merge.list <- list(
  c(4, 5, 9, 20)
)

Idents(seu) <- "seurat_clusters"
orig_levels <- levels(seu)
orig_levels_chr <- as.character(orig_levels)

intermediate_map <- setNames(orig_levels_chr, orig_levels_chr)  

for(group in merge.list){
  group_chr <- as.character(group)
  present <- group_chr[group_chr %in% orig_levels_chr]  
  if(length(present) == 0) next
  rep_label <- present[1]  
  intermediate_map[present] <- rep_label
}

cell_orig <- as.character(Idents(seu))
cell_intermediate <- intermediate_map[cell_orig]

counts <- sort(table(cell_intermediate), decreasing = TRUE)
ordered_intermediate_levels <- names(counts)

final_numbers <- as.character(0:(length(ordered_intermediate_levels)-1))
names(final_numbers) <- ordered_intermediate_levels

final_map_per_original <- final_numbers[intermediate_map]
names(final_map_per_original) <- orig_levels_chr

horizontal_vec <- paste0("c('", paste(final_map_per_original, collapse = "','"), "')")
cat("Final new_cluster_ids (horizontal):\n")
cat(horizontal_vec, "\n\n")

seu <- RenameIdents(seu, final_map_per_original)
seu@meta.data$seurat_clusters <- Idents(seu)

seu$seurat_clusters <- factor(as.character(seu$seurat_clusters),
                              levels = as.character(sort(as.numeric(as.character(unique(seu$seurat_clusters))))))


In [ ]:
### Save UMAP with merged clusters
jpeg(paste0(figDir, "UMAP.final_clusters_full.jpeg"), width = 900, height = 900, res = 150)
DimPlot(seu, reduction = "umap", group.by = "seurat_clusters", label = TRUE, label.size = 5)
dev.off()
DimPlot(seu, reduction = "umap", group.by = "seurat_clusters", label = TRUE, label.size = 5)

plot_height <- length(unique(seu$genotype)) * 2

jpeg(paste0(figDir, "UMAP.final_clusters_split_by_cond.jpeg"), width = 1500, height = 250 * plot_height, res = 150)
DimPlot(seu, reduction = "umap", group.by = "seurat_clusters", label = TRUE, label.size = 5, split.by = "condition", ncol = 2)
dev.off()

In [ ]:
### Keep cells in cluster 15 with at least one hemocyte marker
hemocyte_markers <- c("Hml",          "Pxn",          "crq",          "NimC1",          "NimB4",          "NimB5",          "Col4a1",          "PPO1",          "PPO2",          "lz",          "peb",          "PPO3",          "msn",          "mys",          "cher",          "regucalcin",          "SPARC",          "Npc2a",          "Ppn",          "Inos",          "Nplp2",          "CG14629",          "CG34331",          "srp",          "drpr"
)
cluster_15 <- subset(seu, idents = "15")
counts <- GetAssayData(cluster_15, slot = "counts")
present_markers <- hemocyte_markers[hemocyte_markers %in% rownames(counts)]
if (length(present_markers) == 0) {
  stop("None of the hemocyte markers were found in cluster 15.")
}
marker_expr <- counts[present_markers, , drop = TRUE]
expressed_cells <- colnames(marker_expr)[Matrix::colSums(marker_expr > 0) > 0]
cluster_15 <- subset(cluster_15, cells = expressed_cells)
counts <- GetAssayData(cluster_15, slot = "counts")

seu <- subset(seu, cells = c(Cells(seu)[seu$seurat_clusters != "15"], colnames(cluster_15)))

### save counts from cluster 15
write.csv(as.data.frame(counts), file = paste0(tabDir, "cluster 15 hemocyte marker counts.csv"))

In [ ]:
### Save/load finalised clusters object
# saveRDS(seu, file=paste0(repDir, "final_clusters.rds"))
seu <- readRDS(paste0(repDir, "final_clusters.rds"))

In [ ]:
## Find markers for finalised clusters
# DefaultAssay(seu)<-'SCT'
# Idents(seu) <- paste0('seurat_clusters')
# all.markers <- FindAllMarkers(seu, only.pos = FALSE, min.pct = 0.2, logfc.threshold = 0.5)
# write.csv(all.markers,file=paste0(tabDir,'allMarkers_final_clusters.csv'))
# all.markers %>%
#         group_by(cluster) %>%
#         slice_max(n = 10, order_by = avg_log2FC) -> top
# write.csv(top,file=paste0(tabDir,'top10Markers_final_clusters.csv'))

DefaultAssay(seu) <- 'RNA'
seu <- ScaleData(seu)
genes <- unique(top$gene)
n_clusters <- length(unique(seu$seurat_clusters))
jpeg(paste0(figDir,'top10markers.heatmap_final_clusters.jpeg'), quality = 100, width = 2500, height = 1300, res = 150)
print(DoHeatmap(subset(seu, downsample = 300), features = genes) + NoLegend())
dev.off()

plot_width <- max(8, 0.2 * length(genes))
plot_height <- max(4, 0.3 * n_clusters)
p <-  DotPlot(seu, features = genes, dot.scale = 6) + 
  RotatedAxis() +
  theme(
    axis.text.x = element_text(size = 8)
  ) +
  scale_color_gradientn(
    colours = c("grey", "forestgreen")#,
    # limits = c(0, 1.5),
    # oob = scales::squish
  ) +
  ggtitle("Top 10 Markers per cluster")
ggsave(filename = paste0(figDir, 'top10markers.dotplot_final_clusters.jpeg'), plot = p, width = plot_width, height = plot_height, dpi = 300)
